In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import lightning as L

L.seed_everything(42)


Seed set to 42


42

In [3]:
import warnings
from datamodules.data_classes import GridDataModule, PotsdamVaihingen
from rasterio.errors import NotGeoreferencedWarning

warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)


train_vaihingen_image_glob = "../Vaihingen_dataset/train/images/*.tif"
train_vaihingen_mask_glob = "../Vaihingen_dataset/train/masks/*.tif"
val_vaihingen_image_glob = "../Vaihingen_dataset/val/images/*.tif"
val_vaihingen_mask_glob = "../Vaihingen_dataset/val/masks/*.tif"

vaihingen_dm = GridDataModule(
    dataset_cls=PotsdamVaihingen,
    train_dataset_kwargs={
        "image_glob": train_vaihingen_image_glob,
        "mask_glob": train_vaihingen_mask_glob,
        "reduce_mask": True,
    },
    val_dataset_kwargs={
        "image_glob": val_vaihingen_image_glob,
        "mask_glob": val_vaihingen_mask_glob,
        "reduce_mask": True,
    },
    loader_kwargs={
        "batch_size": None,
        "num_workers": 0,
    },
)


In [4]:
from segmentation_models_pytorch import Unet

In [ ]:
import torch
from torch.utils.data import DataLoader
import torchio as tio
import kornia.augmentation as K
from torchmetrics.segmentation import MeanIoU

from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint


class GridTrainingSegmentator(L.LightningModule):
    def __init__(
        self,
        model,
        criterion,
        loader_kwargs,
        grid_kwargs,
        augmentation=None,
        metric=MeanIoU(num_classes=6),
    ):
        super().__init__()
        self.model = model
        self.criterion = criterion
        self.patch_size = grid_kwargs.get("patch_size")
        self.overlap = grid_kwargs.get("overlap")
        self.loader_kwargs = loader_kwargs
        self.augmentation = augmentation
        self.metric = metric
        ## Save HPs
        self.save_hyperparameters(
            ignore=["model", "criterion", "augmentation", "metric"]
        )

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        image, mask = batch["image"].unsqueeze(-1), batch["mask"].unsqueeze(-1)
        subject = tio.Subject(
            image=tio.ScalarImage(tensor=image), mask=tio.LabelMap(tensor=mask)
        )
        sampler = tio.GridSampler(
            subject,
            patch_size=self.patch_size,
            patch_overlap=self.overlap,
        )

        loader = DataLoader(
            sampler,
            **self.loader_kwargs,
        )
        total_loss = 0
        n_patches = 0
        for patch in loader:
            X, y = (
                patch["image"][tio.DATA].to(self.device),
                patch["mask"][tio.DATA].to(self.device),
            )
            X = X.squeeze(-1)
            y = y.squeeze(-1)

            if self.augmentation is not None:
                X = self.augmentation["intensity_augmentation"](X)
                X, y = self.augmentation["geom_augmentation"](X, y)

            y = y.squeeze(1).long()

            pred = self(X)
            loss = self.criterion(pred, y)
            total_loss += loss
            n_patches += 1

        avg_loss = total_loss / n_patches
        self.log(
            "train_loss",
            avg_loss.item(),
            on_step=True,
            on_epoch=True,
            prog_bar=True,
            batch_size=1,
            logger=True,
        )

        return avg_loss

    def validation_step(self, batch, batch_idx):
        image, mask = batch["image"].unsqueeze(-1), batch["mask"].unsqueeze(-1)
        subject = tio.Subject(
            image=tio.ScalarImage(tensor=image), mask=tio.LabelMap(tensor=mask)
        )
        sampler = tio.GridSampler(
            subject,
            patch_size=self.patch_size,
            patch_overlap=self.overlap,
        )
        aggregator = tio.GridAggregator(sampler)
        with torch.no_grad():
            n_patches = 0
            for patch in DataLoader(sampler, **self.loader_kwargs):
                X, y = (
                    patch["image"][tio.DATA].to(self.device),
                    patch["mask"][tio.DATA].to(self.device),
                )
                X = X.squeeze(-1)
                y = y.squeeze(-1).squeeze(1).long()

                pred = self(X)
                aggregator.add_batch(
                    pred.detach().cpu().unsqueeze(-1), patch[tio.LOCATION]
                )
                n_patches += 1

        full_pred = aggregator.get_output_tensor().squeeze(-1)

        y_hat = full_pred.argmax(dim=0).unsqueeze(0).to(self.device)
        gt = mask.squeeze(-1).long()

        self.metric(y_hat, gt)
        return None

    def on_fit_start(self):
        self.metric = self.metric.to(self.device)

    def on_validation_epoch_end(self):
        miou = self.metric.compute()
        self.log("val_iou", miou, on_epoch=True, prog_bar=True, batch_size=1)
        self.metric.reset()

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-4)


model = GridTrainingSegmentator(
    Unet("resnet34", in_channels=3, classes=6),
    criterion=torch.nn.CrossEntropyLoss(),
    grid_kwargs={"patch_size": (256, 256, 1), "overlap": (32, 32, 0)},
    loader_kwargs={"batch_size": 1, "num_workers": 0},
    augmentation=dict(
        geom_augmentation=K.AugmentationSequential(
            K.RandomHorizontalFlip(p=0.5),
            K.RandomVerticalFlip(p=0.5),
            data_keys=["input", "mask"],
        ),
        intensity_augmentation=K.AugmentationSequential(
            K.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.8),
            K.RandomGaussianNoise(mean=0.0, std=0.05, p=0.3),
            data_keys=["input"],  # solo imagen
        ),
    ),
    metric=MeanIoU(num_classes=6),
)


checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints/vaihingen_segmentation/version_0/checkpoints",
    filename="epoch-{epoch:02d}-step-{step}",
    save_top_k=1,  # keep best 3 models
    monitor="val_iou",  # or "val_loss"
    mode="max",  # IoU → maximize
    save_last=True,  # VERY IMPORTANT for resume
)
logger = TensorBoardLogger(save_dir="logs", version=1, name="minimal_example")
trainer = L.Trainer(
    max_epochs=1,
    logger=logger,
    callbacks=[checkpoint_callback],
    accelerator="cpu",
    devices=1,
    enable_progress_bar=True,
    deterministic=False,  ## Not possible for CE
    check_val_every_n_epoch=1,
    fast_dev_run=False,  # Para probar que el código corre sin errores, luego quitarlo para entrenar en serio
    overfit_batches=0,
    log_every_n_steps=1,
)

trainer.fit(model, vaihingen_dm)

GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/home/datacuber/Documents/semantic_segmentation/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ Unet             │ 24.4 M │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
│ 2 │ metric    │ MeanIoU          │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 24.4 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.4 M                                                                                               
Total estimated model params size (MB): 97                                                                         
Modules in train mode: 190                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/datacuber/Documents/semantic_segmentation/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/_pyt
ree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and 
treespec.is_leaf()` instead.

/home/datacuber/Documents/semantic_segmentation/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connec
tors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider 
increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

/home/datacuber/Documents/semantic_segmentation/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connec
tors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider 
increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.